# TableRAG MVP: A Prototype for Table Question Answering

This notebook builds a small, step-by-step TableRAG prototype inspired by [*TableRAG: Million-Token Table Understanding with Language Models*](https://arxiv.org/abs/2410.04739).

**Goal:** prepare a table for question answering without placing every row in an LLM prompt. TableRAG retrieves relevant column information and cell values, then gives that evidence to a solver.

## What this notebook builds

1. Load a table and keep the original data available.
2. Create a compact schema profile for its columns.
3. Build a frequency-ranked catalog of distinct column-value pairs, capped by a configurable budget.

In [127]:
import os
import json
import pandas as pd
import torch

from pathlib import Path
from sklearn.datasets import fetch_openml
from transformers import AutoModelForCausalLM, AutoTokenizer

## 1. Load and Inspect the Titanic Data

We use the Titanic dataset identified by OpenML dataset ID `40945` as a small development table. The first run downloads it; subsequent runs can load the saved Parquet copy.

In [128]:
path = 'titanic.parquet'

path = Path("titanic.parquet")

if path.exists():
    df = pd.read_parquet(path)
else:
    data = fetch_openml(data_id=40945, as_frame=True)
    df = data.frame
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path)

display(df.shape)
display(df.head())

(1309, 14)

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1.0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1.0,1,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1.0,0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1.0,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1.0,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## 2. Build the Schema Profile

The schema profile records each column's name, an automatically inferred type, and compact examples. It follows the paper's approach: numeric and datetime columns are summarized by their minimum and maximum, while categorical columns show up to three of their most frequent values.

Missing values are excluded when calculating examples. The type inference is a first pass based on pandas dtypes, so it may not capture meaning on its own: for example, `pclass` is stored numerically but represents ticket classes.

In [129]:
def build_schema_metadata(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for column in df.columns:
        series = df[column]
        values = series.dropna() # Remove any None-values

        if pd.api.types.is_datetime64_any_dtype(series):
            column_type = "datetime"
            examples = (
                {"min": values.min(), "max": values.max()}
                if not values.empty else {}
            )
        elif pd.api.types.is_numeric_dtype(series):
            column_type = "numeric"
            examples = (
                {"min": values.min(), "max": values.max()}
                if not values.empty else {}
            )
        else:
            column_type = "categorical"
            examples = [
                {"value": value, "count": int(count)}
                for value, count in values.value_counts().head(3).items()
            ]

        rows.append({
            "column": column,
            "type": column_type,
            "examples": examples,
        })

    return pd.DataFrame(rows).set_index("column")

schema_metadata = build_schema_metadata(df)
display(schema_metadata)

,type,examples
column,,
pclass,numeric,"{'min': 1.0, 'max': 3.0}"
survived,categorical,"[{'value': '0', 'count': 809}, {'value': '1', ..."
name,categorical,"[{'value': 'Connolly, Miss. Kate', 'count': 2}..."
sex,categorical,"[{'value': 'male', 'count': 843}, {'value': 'f..."
age,numeric,"{'min': 0.1667, 'max': 80.0}"
sibsp,numeric,"{'min': 0.0, 'max': 8.0}"
parch,numeric,"{'min': 0.0, 'max': 9.0}"
ticket,categorical,"[{'value': 'CA. 2343', 'count': 11}, {'value':..."
fare,numeric,"{'min': 0.0, 'max': 512.3292}"


## 3. Build the Frequency-Ranked Cell Catalog

For every column, count each distinct non-missing value and store it as a `(column, value, frequency)` entry. Entries are ranked by frequency, and the catalog is limited to the first `B` pairs, following the paper's frequency-aware cell indexing approach.

In [130]:
B = 10000 # Budget outlined in the paper

pairs = []

for col in df.columns:
    counts = df[col].value_counts(dropna=True) # returns a series containing the frequency of each unique value in col

    for val, freq in counts.items():
        if freq > 0:
            pairs.append({
                "column": col,
                "value": val,
                "frequency": int(freq)
            })

cell_catalog = (
    pd.DataFrame(pairs)
    .sort_values("frequency", ascending=False, kind="stable")
    .head(B)
    .reset_index(drop=True)
    .set_index("column")
)

display(cell_catalog.head(10))
print(f"Catalog entries: {len(cell_catalog)} (budget: {B})")


,value,frequency
column,,
parch,0.0,1002
embarked,S,914
sibsp,0.0,891
sex,male,843
survived,0,809
pclass,3.0,709
survived,1,500
sex,female,466
pclass,1.0,323


Catalog entries: 3343 (budget: 10000)


## 4. Query Expansion

In [131]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "mps" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype)
model.to(device)
model.eval()


Loading weights: 100%|██████████| 290/290 [00:02<00:00, 112.06it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [132]:
def parse_query_list(raw: str) -> list[str]:
    text = raw.strip()

    if text.startswith("```"):
        lines = text.splitlines()
        if len(lines) < 3 or not lines[-1].strip().startswith("```"):
            raise ValueError("The model response has an unclosed code fence.")
        text = "\n".join(lines[1:-1]).strip()

    try:
        result = json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError("The model response was not a valid JSON list.") from exc

    if (
        not isinstance(result, list)
        or any(not isinstance(item, str) or not item.strip() for item in result)
    ):
        raise ValueError("Expected a JSON list of non-empty strings.")

    return [item.strip() for item in result]

In [133]:
TABLE_DESCRIPTION = "The original Titanic dataset, describing the survival status of individual passengers on the Titanic"
QUERY = "Did Allen, Miss. Elisabeth Walton survive?"


QUERY_EXPANSION_SCHEMA_RETRIEVAL = f"""
Given a large table regarding "{TABLE_DESCRIPTION}", I want to answer a question: "{QUERY}"
Since I cannot view the table directly, please suggest some column names that might contain
the necessary data to answer this question.
Please answer with a list of column names in JSON format without any additional explanation.
Example:
["column1", "column2", "column3"]
"""

QUERY_EXPANSION_CELL_RETRIEVAL = f"""
Given a large table regarding "{TABLE_DESCRIPTION}", I want to answer a question: "{QUERY}"
Please extract some keywords which might appear in the table cells and help answer the question.
The keywords should be categorical values rather than numerical values.
The keywords should be contained in the question.
Please answer with a list of keywords in JSON format without any additional explanation.
Example:
["keyword1", "keyword2", "keyword3"]
"""

print(QUERY_EXPANSION_SCHEMA_RETRIEVAL)
print(QUERY_EXPANSION_CELL_RETRIEVAL)


Given a large table regarding "The original Titanic dataset, describing the survival status of individual passengers on the Titanic", I want to answer a question: "Did Allen, Miss. Elisabeth Walton survive?"
Since I cannot view the table directly, please suggest some column names that might contain
the necessary data to answer this question.
Please answer with a list of column names in JSON format without any additional explanation.
Example:
["column1", "column2", "column3"]


Given a large table regarding "The original Titanic dataset, describing the survival status of individual passengers on the Titanic", I want to answer a question: "Did Allen, Miss. Elisabeth Walton survive?"
Please extract some keywords which might appear in the table cells and help answer the question.
The keywords should be categorical values rather than numerical values.
The keywords should be contained in the question.
Please answer with a list of keywords in JSON format without any additional explanation.
E

In [134]:
def llm_raw_response(query_expansion_prompt):
    messages = [
        {
            "role": "system",
            "content": query_expansion_prompt
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(device)

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw_response = tokenizer.decode(
        output_ids[0, input_length:],
        skip_special_tokens=True,
    )

    return raw_response


In [135]:
raw_response_schema = llm_raw_response(QUERY_EXPANSION_SCHEMA_RETRIEVAL)
raw_response_cell = llm_raw_response(QUERY_EXPANSION_CELL_RETRIEVAL)

print(raw_response_schema)
print(raw_response_cell)

```json
["Survived", "Name", "Sex", "Age", "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"]
```
```json
["survived", "Allen", "Miss. Elisabeth Walton"]
```


In [136]:
parsed_response_schema = parse_query_list(raw_response_schema)
parsed_response_cell = parse_query_list(raw_response_cell)

print(parsed_response_schema)
print(parsed_response_cell)

['Survived', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
['survived', 'Allen', 'Miss. Elisabeth Walton']
